# Customer Churn Data Pipeline

## 1. Upload & Detect Files

In [ ]:
import pandas as pd
import numpy as np
import os

try:
    from google.colab import files
    print('Upload CSV files')
    uploaded = files.upload()
except:
    print('Running locally')

files_list = os.listdir()
print(files_list)

## 2. Auto Detect Files

In [ ]:
def find_file(keywords):
    for f in os.listdir():
        if all(k in f.lower() for k in keywords):
            return f
    return None

file_map = {
    'customers': ['customers'],
    'orders': ['orders'],
    'support': ['support'],
    'web': ['web'],
    'churn': ['churn'],
    'campaign': ['intervention']
}

detected = {k: find_file(v) for k, v in file_map.items()}
print(detected)

## 3. Load Data

In [ ]:
missing = [k for k,v in detected.items() if v is None]
if missing:
    raise Exception(f'Missing files: {missing}')

customers = pd.read_csv(detected['customers'])
orders = pd.read_csv(detected['orders'])
support = pd.read_csv(detected['support'])
web = pd.read_csv(detected['web'])
churn = pd.read_csv(detected['churn'])
campaign = pd.read_csv(detected['campaign'])

print(customers.shape, orders.shape)

## 4. Convert Dates

In [ ]:
def convert_dates(df):
    for col in df.columns:
        if 'date' in col.lower() or 'time' in col.lower():
            try:
                df[col] = pd.to_datetime(df[col])
            except:
                pass
    return df

orders = convert_dates(orders)
support = convert_dates(support)
web = convert_dates(web)
campaign = convert_dates(campaign)
churn = convert_dates(churn)

## 5. Feature Engineering

In [ ]:
orders['order_date'] = pd.to_datetime(orders['order_date'])
order_agg = orders.groupby('customer_id').agg({
    'order_id': 'count',
    'gross_amount': ['sum','mean']
}).reset_index()
order_agg.columns = ['customer_id','total_orders','total_spend','avg_order_value']

## 6. Merge

In [ ]:
df_final = churn.copy()
df_final = df_final.merge(customers, on='customer_id', how='left')
df_final = df_final.merge(order_agg, on='customer_id', how='left')
print(df_final.head())